In [1]:
# ==========================================
# Author: George Ayensu, PE
# Project: PhD Working Code - Week 2 (Data Center Optimization)
# Case Study: Proof of Transferability (GWL to Data Centers)
# Date: November 2026
# ==========================================

!pip install pyomo highspy -q
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=range(1, 25))

# --- REAL-WORLD DATA CENTER PARAMETERS ---
# Cooling Load (kW_thermal): High during day (10,000 kW), drops at night (6,000 kW)
def load_rule(model, t):
    if t >= 22 or t <= 4: return 6000.0  # Night drop
    else: return 10000.0                 # Day peak
model.cooling_load = pyo.Param(model.T, initialize=load_rule)

# Chillers (Max Cooling Capacity in kW_thermal)
max_old_chiller = 4000.0   # Older, less efficient chiller
max_new_chiller = 8000.0   # Newer, high-efficiency chiller

# Efficiency (kW_electric consumed per kW_thermal cooling provided)
# Old Chiller: 0.35 (needs 1.4 MW electricity to provide 4 MW cooling)
# New Chiller: 0.20 (needs 1.6 MW electricity to provide 8 MW cooling)
old_efficiency = 0.35
new_efficiency = 0.20

# Thermal Energy Storage (TES) Tank (Stores chilled water)
max_tes_capacity = 50000.0 # kWh_thermal

# Tariff (Hypothetical: Off-peak $0.10, Peak $0.25)
def price_rule(model, t):
    if t <= 6 or t >= 22: return 0.10
    else: return 0.25
model.price = pyo.Param(model.T, initialize=price_rule)

# Variables
model.old_chiller = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.new_chiller = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.tes_level = pyo.Var(model.T, within=pyo.NonNegativeReals)

# Constraints
# 1. Chiller Capacity Limits
model.limit_old = pyo.Constraint(model.T, rule=lambda m, t: m.old_chiller[t] <= max_old_chiller)
model.limit_new = pyo.Constraint(model.T, rule=lambda m, t: m.new_chiller[t] <= max_new_chiller)

# 2. Thermal Energy Storage (TES) Capacity Limit
model.tes_cap = pyo.Constraint(model.T, rule=lambda m, t: m.tes_level[t] <= max_tes_capacity)

# 3. Thermal Balance: Supply (Chillers + TES discharge) == Load
def tes_balance_rule(model, t):
    if t == 1:
        return model.tes_level[t] == 0 + model.old_chiller[t] + model.new_chiller[t] - model.cooling_load[t]
    else:
        return model.tes_level[t] == model.tes_level[t-1] + model.old_chiller[t] + model.new_chiller[t] - model.cooling_load[t]
model.balance = pyo.Constraint(model.T, rule=tes_balance_rule)

# Objective: Minimize Electricity Cost (Tariff * Chiller Electrical Draw)
model.cost = pyo.Objective(
    expr=sum(model.price[t] * ((model.old_chiller[t] * old_efficiency) + (model.new_chiller[t] * new_efficiency)) for t in model.T),
    sense=pyo.minimize
)

# Solve
solver = pyo.SolverFactory('appsi_highs')
solver.solve(model)

print(f"Total Daily Electricity Cost (Data Center): ${model.cost.expr():,.2f}")
print("-------------------------------------------------------------------------")
print("Hour | Price | Cooling Load | Old Chiller | New Chiller | TES Level (kWh)")
for t in model.T:
    print(f"{t:4} | ${model.price[t]:4.2f} | {model.cooling_load[t]:12.1f} | {model.old_chiller[t].value:11.1f} | {model.new_chiller[t].value:11.1f} | {model.tes_level[t].value:14.1f}")


Total Daily Electricity Cost (Data Center): $8,335.00
-------------------------------------------------------------------------
Hour | Price | Cooling Load | Old Chiller | New Chiller | TES Level (kWh)
   1 | $0.10 |       6000.0 |      4000.0 |      8000.0 |         6000.0
   2 | $0.10 |       6000.0 |      4000.0 |      8000.0 |        12000.0
   3 | $0.10 |       6000.0 |      4000.0 |      8000.0 |        18000.0
   4 | $0.10 |       6000.0 |      4000.0 |      8000.0 |        24000.0
   5 | $0.10 |      10000.0 |      4000.0 |      8000.0 |        26000.0
   6 | $0.10 |      10000.0 |      4000.0 |      8000.0 |        28000.0
   7 | $0.25 |      10000.0 |         0.0 |      8000.0 |        26000.0
   8 | $0.25 |      10000.0 |         0.0 |      8000.0 |        24000.0
   9 | $0.25 |      10000.0 |         0.0 |      8000.0 |        22000.0
  10 | $0.25 |      10000.0 |         0.0 |      8000.0 |        20000.0
  11 | $0.25 |      10000.0 |         0.0 |      8000.0 |        180